# Notebook 1: Exploratory Data Analysis (EDA)
## Predicting Social Media Content Virality Using Distributed ML on Reddit Big Data

**Dataset**: webis/tldr-17 (~18GB, ~3M+ Reddit posts from HuggingFace)  
**Task**: Binary Classification — Predict whether a Reddit post is "viral" or "non-viral"  
**Tools**: PySpark, Matplotlib, Seaborn

In [1]:
# ============================================================================
# Cell 1: Setup & Imports
# ============================================================================
import os
import sys
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import *

warnings.filterwarnings('ignore')

# Resolve project root robustly whether notebook is run from repo root or notebooks/
cwd = os.getcwd()
candidate_roots = [
    cwd,
    os.path.abspath(os.path.join(cwd, '..')),
]
PROJECT_ROOT = next(
    (p for p in candidate_roots if os.path.exists(os.path.join(p, 'data', 'raw'))),
    os.path.abspath(os.path.join(cwd, '..'))
)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

RAW_CSV = os.path.join(PROJECT_ROOT, 'data', 'raw', 'reddit_posts.csv')
TABLEAU_DIR = os.path.join(PROJECT_ROOT, 'tableau')
TABLEAU_DATA_DIR = os.path.join(TABLEAU_DIR, 'data')
os.makedirs(TABLEAU_DIR, exist_ok=True)
os.makedirs(TABLEAU_DATA_DIR, exist_ok=True)

sns.set_theme(style='whitegrid', palette='viridis', font_scale=1.2)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print(f'Project root: {PROJECT_ROOT}')
print(f'Raw CSV path: {RAW_CSV}')

Project root: c:\Users\shiva\OneDrive\Desktop\Predicting Social Media Content Virality Using Distributed Machine Learning on Reddit Big Data\reddit-virality-project
Raw CSV path: c:\Users\shiva\OneDrive\Desktop\Predicting Social Media Content Virality Using Distributed Machine Learning on Reddit Big Data\reddit-virality-project\data\raw\reddit_posts.csv


In [ ]:
# ============================================================================
# Cell 2: Create Spark Session
# ============================================================================
spark = (
    SparkSession.builder
    .appName('RedditVirality_EDA')
    .master('local[*]')
    .config('spark.driver.memory', '8g')
    .config('spark.sql.shuffle.partitions', 200)
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.driver.maxResultSize', '4g')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')
print(f'Spark UI available at: http://localhost:4040')

In [ ]:
# ============================================================================
# Cell 3: Load Dataset
# ============================================================================
if not os.path.exists(RAW_CSV):
    raise FileNotFoundError(f'Dataset not found at: {RAW_CSV}')

print(f'Loading data from: {RAW_CSV}')
print(f'File size: {os.path.getsize(RAW_CSV) / (1024**3):.2f} GB')

df = spark.read.csv(
    RAW_CSV,
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"',
    mode='PERMISSIVE'
 )

print('\nDataset loaded successfully.')

## 1. Dataset Overview
Understand the size, schema, and basic statistics of the Reddit dataset.

In [ ]:
# ============================================================================
# Cell 4: Basic Dataset Statistics
# ============================================================================
total_rows = df.count()
total_cols = len(df.columns)

print('=' * 60)
print('DATASET OVERVIEW')
print('=' * 60)
print(f'Total rows:    {total_rows:,}')
print(f'Total columns: {total_cols}')
print(f'File size:     {os.path.getsize(RAW_CSV) / (1024**3):.2f} GB')
print(f'\nColumn names:  {df.columns}')
print(f'\nSchema:')
df.printSchema()

In [ ]:
# ============================================================================
# Cell 5: Sample Rows
# ============================================================================
print('First 5 rows (sample):')
df.show(5, truncate=80)

In [ ]:
# ============================================================================
# Cell 6: Missing Value Analysis
# ============================================================================
print('MISSING VALUE ANALYSIS')
print('=' * 60)

# Compute all missing counts in one pass for better performance on large data
missing_exprs = [
    F.sum(
        F.when(
            F.col(col_name).isNull() | (F.trim(F.col(col_name).cast('string')) == ''),
            1
        ).otherwise(0)
    ).alias(col_name)
    for col_name in df.columns
]

missing_row = df.select(*missing_exprs).first().asDict()
missing_counts = []
for col_name in df.columns:
    null_count = int(missing_row.get(col_name, 0))
    pct = (null_count / total_rows) * 100 if total_rows else 0.0
    missing_counts.append({'Column': col_name, 'Missing': null_count, 'Pct': pct})
    print(f'  {col_name:20s} -> {null_count:>10,} missing ({pct:.2f}%)')

missing_df = pd.DataFrame(missing_counts).sort_values('Pct', ascending=False)

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(
    missing_df['Column'][::-1],
    missing_df['Pct'][::-1],
    color=sns.color_palette('viridis', len(missing_df))
)
ax.set_xlabel('Missing Percentage (%)')
ax.set_title('Missing Values by Column')
for bar, pct in zip(bars, missing_df['Pct'][::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{pct:.1f}%', va='center', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(TABLEAU_DIR, 'missing_values.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2. Subreddit Distribution Analysis
Analyze the distribution of posts across subreddits to define our virality threshold.

In [ ]:
# ============================================================================
# Cell 7: Subreddit Distribution (Top 30)
# ============================================================================
subreddit_counts = (
    df.groupBy('subreddit')
    .count()
    .orderBy(F.desc('count'))
)

total_subreddits = subreddit_counts.count()
print(f'Total unique subreddits: {total_subreddits:,}')

# Get top 30 for plotting
top30 = subreddit_counts.limit(30).toPandas()

fig, ax = plt.subplots(figsize=(14, 8))
ax.barh(top30['subreddit'][::-1], top30['count'][::-1], color=sns.color_palette('viridis', 30))
ax.set_xlabel('Number of Posts')
ax.set_title('Top 30 Subreddits by Post Count')
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'tableau', 'top30_subreddits.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 subreddits:')
subreddit_counts.show(10)

In [ ]:
# ============================================================================
# Cell 8: Define Virality Label
# ============================================================================
# Strategy: Posts in subreddits above 80th percentile post count = 'viral'
# Rationale: Higher-activity subreddits have wider audience reach

subreddit_stats = subreddit_counts.toPandas()

threshold = subreddit_stats['count'].quantile(0.80)
print(f'80th percentile post count threshold: {threshold:,.0f}')

viral_subreddits = set(subreddit_stats[subreddit_stats['count'] >= threshold]['subreddit'].tolist())
print(f'Number of viral subreddits: {len(viral_subreddits)}')
print(f'Number of non-viral subreddits: {total_subreddits - len(viral_subreddits)}')

# Create virality label
viral_list = list(viral_subreddits)
df_labeled = df.withColumn(
    'is_viral',
    F.when(F.col('subreddit').isin(viral_list), 1).otherwise(0)
)

# Class distribution
class_dist = df_labeled.groupBy('is_viral').count().orderBy('is_viral').toPandas()
class_dist_map = dict(zip(class_dist['is_viral'].astype(int), class_dist['count'].astype(int)))
print('\nClass Distribution:')
print(class_dist)

# Pie chart (consistent label ordering)
pie_values = [class_dist_map.get(0, 0), class_dist_map.get(1, 0)]
labels = ['Non-Viral (0)', 'Viral (1)']
colors = ['#3498db', '#e74c3c']

fig, ax = plt.subplots(figsize=(8, 8))
ax.pie(
    pie_values,
    labels=labels,
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    textprops={'fontsize': 14}
 )
ax.set_title('Class Distribution: Viral vs Non-Viral Posts', fontsize=16)
plt.savefig(os.path.join(TABLEAU_DIR, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3. Text Length Analysis

In [ ]:
# ============================================================================
# Cell 9: Text Length Distributions
# ============================================================================
df_lengths = df_labeled.withColumn('body_length', F.length('body')) \
    .withColumn('summary_length', F.length('summary')) \
    .withColumn('word_count', F.size(F.split('body', r'\s+')))

# Descriptive statistics
print('Body Length Statistics:')
df_lengths.select('body_length', 'summary_length', 'word_count').describe().show()

# Sample for plotting
plot_sample = df_lengths.select('body_length', 'summary_length', 'word_count', 'is_viral') \
    .sample(False, 0.01, seed=42).toPandas()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Body length distribution
axes[0].hist(plot_sample[plot_sample['body_length'] < 5000]['body_length'], 
             bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Body Length Distribution')
axes[0].set_xlabel('Characters')
axes[0].set_ylabel('Frequency')

# Summary length distribution
axes[1].hist(plot_sample[plot_sample['summary_length'] < 1000]['summary_length'], 
             bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_title('Summary Length Distribution')
axes[1].set_xlabel('Characters')

# Word count distribution
axes[2].hist(plot_sample[plot_sample['word_count'] < 1000]['word_count'], 
             bins=50, color='green', edgecolor='black', alpha=0.7)
axes[2].set_title('Word Count Distribution')
axes[2].set_xlabel('Words')

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'tableau', 'text_length_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# Cell 10: Text Length by Virality Class (Box Plot)
# ============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Filter extreme outliers for better visualization
sample = plot_sample[plot_sample['body_length'] < 5000].copy()
sample['is_viral'] = sample['is_viral'].map({0: 'Non-Viral', 1: 'Viral'})

sns.boxplot(data=sample, x='is_viral', y='body_length', ax=axes[0], palette=['#3498db', '#e74c3c'])
axes[0].set_title('Body Length by Virality')

sample2 = plot_sample[plot_sample['summary_length'] < 1000].copy()
sample2['is_viral'] = sample2['is_viral'].map({0: 'Non-Viral', 1: 'Viral'})
sns.boxplot(data=sample2, x='is_viral', y='summary_length', ax=axes[1], palette=['#3498db', '#e74c3c'])
axes[1].set_title('Summary Length by Virality')

sample3 = plot_sample[plot_sample['word_count'] < 1000].copy()
sample3['is_viral'] = sample3['is_viral'].map({0: 'Non-Viral', 1: 'Viral'})
sns.boxplot(data=sample3, x='is_viral', y='word_count', ax=axes[2], palette=['#3498db', '#e74c3c'])
axes[2].set_title('Word Count by Virality')

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'tableau', 'text_length_by_virality.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Correlation Analysis

In [ ]:
# ============================================================================
# Cell 11: Feature Correlation Heatmap
# ============================================================================
# Engineer basic numeric features for correlation
df_feat = df_lengths.withColumn('has_url', 
    F.when(F.col('body').rlike('https?://'), 1).otherwise(0)) \
    .withColumn('question_marks', 
        (F.length('body') - F.length(F.regexp_replace('body', r'\?', ''))).cast('int')) \
    .withColumn('exclamation_marks',
        (F.length('body') - F.length(F.regexp_replace('body', '!', ''))).cast('int'))

corr_sample = df_feat.select(
    'body_length', 'summary_length', 'word_count', 
    'has_url', 'question_marks', 'exclamation_marks', 'is_viral'
).sample(False, 0.01, seed=42).toPandas()

# Correlation matrix
corr_matrix = corr_sample.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, square=True, linewidths=1, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'tableau', 'correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Summary Statistics Export for Tableau

In [ ]:
# ============================================================================
# Cell 12: Export EDA Statistics for Tableau
# ============================================================================
os.makedirs(TABLEAU_DATA_DIR, exist_ok=True)

# Export subreddit distribution
subreddit_stats.to_csv(
    os.path.join(TABLEAU_DATA_DIR, 'subreddit_distribution.csv'),
    index=False
)

# Export class distribution
class_dist.to_csv(
    os.path.join(TABLEAU_DATA_DIR, 'class_distribution.csv'),
    index=False
)

# Export missing values info
missing_df.to_csv(
    os.path.join(TABLEAU_DATA_DIR, 'missing_values.csv'),
    index=False
)

# Export correlation data
corr_matrix.to_csv(
    os.path.join(TABLEAU_DATA_DIR, 'correlation_matrix.csv')
)

# Export sample features for Tableau
corr_sample.to_csv(
    os.path.join(TABLEAU_DATA_DIR, 'eda_features_sample.csv'),
    index=False
)

print('All EDA data exported to tableau/data/')
print('Files:', os.listdir(TABLEAU_DATA_DIR))

In [ ]:
# ============================================================================
# Cell 13: EDA Summary
# ============================================================================
class_0_count = int(class_dist_map.get(0, 0))
class_1_count = int(class_dist_map.get(1, 0))

print('=' * 60)
print('EDA SUMMARY')
print('=' * 60)
print('Dataset: webis/tldr-17 (Reddit TLDR posts)')
print('Source: HuggingFace Datasets')
print(f'Total rows: {total_rows:,}')
print(f'Total columns: {total_cols}')
print(f'File size: {os.path.getsize(RAW_CSV) / (1024**3):.2f} GB')
print(f'Total unique subreddits: {total_subreddits:,}')
print(f'Virality threshold (80th pct): {threshold:,.0f} posts')
print(f'Viral subreddits: {len(viral_subreddits)}')
print(f'Class 0 (non-viral): {class_0_count:,}')
print(f'Class 1 (viral): {class_1_count:,}')
print('=' * 60)

spark.stop()
print('\nSpark session stopped. EDA complete.')